# dislop T3 Local Small Language Model Benchmark (Colab GPU)

Benchmarks optional local GGUF/Ollama models (`qwen3:0.6b`, `qwen3:1.7b`, `qwen3:4b`, `gemma3:1b`, `gemma3:4b`) via `scripts/run_live_model_bench.mjs` using the single shared `scoreRevision()` scorer from `scripts/run_benchmark.mjs` across all 18 **Suite B** tasks (`revision-v1.json` + `revision-hard-v2.json`).

In [ ]:
#@title 1. Check GPU, Node.js 20+, and clone dislop (pinned ref)
import os, subprocess, shutil

DISLOP_REF = "main"  #@param {type:"string"}
subprocess.run(["nvidia-smi"], check=False)

if not shutil.which("node") or int(subprocess.check_output(["node", "-v"], text=True).strip().lstrip("v").split(".")[0]) < 20:
    subprocess.run(["bash", "-lc", "curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs"], check=True)

if not os.path.exists("/content/dislop"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", DISLOP_REF, "https://github.com/dislop-ai/dislop.git", "/content/dislop"], check=True)

os.chdir("/content/dislop")
print("Node:", subprocess.check_output(["node", "-v"], text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
#@title 2. Start Ollama on Colab GPU and pull T3 candidate models
import subprocess, time, urllib.request

MODELS = [
    "qwen3:0.6b",
    "qwen3:1.7b",
    "qwen3:4b",
    "gemma3:1b",
    "gemma3:4b",
]

if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
    subprocess.run(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"], check=True)

server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=1)
        break
    except Exception:
        time.sleep(1)

for model in MODELS:
    print(f"Pulling {model}...")
    subprocess.run(["ollama", "pull", model], check=True)
print("All candidate models ready.")

In [ ]:
#@title 3. Run unified Suite B live benchmark runner (18 tasks, bare vs. fact_locked)
import json, subprocess
from pathlib import Path

OUT_PATH = Path("/content/t3-local-slm-colab.json")
REPEATS = 1  #@param {type:"integer"}

cmd = [
    "node", "scripts/run_live_model_bench.mjs",
    "--models", ",".join(MODELS),
    "--arms", "bare,fact_locked",
    "--repeats", str(REPEATS),
    "--out", str(OUT_PATH),
]
subprocess.run(cmd, check=True)

# Attach Ollama model digests for reproducibility
report = json.loads(OUT_PATH.read_text(encoding="utf-8"))
tags = subprocess.check_output(["ollama", "list"], text=True)
report["ollamaList"] = tags.strip()
OUT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("Saved unified dislop.live-bench/1 report:", OUT_PATH)

In [ ]:
#@title 4. Print leaderboard & optionally sync JSON to Google Drive
import json
from pathlib import Path

report = json.loads(Path("/content/t3-local-slm-colab.json").read_text(encoding="utf-8"))
print("| Model | Arm | Avg Score | All-Clear | Truncated | Median Latency |")
print("|---|---|---:|---:|---:|---:|")
for r in report.get("results", []):
    print(f"| `{r['model']}` | `{r['arm']}` | {r['avgScore']}/100 | {r['allClear']}/{r['total']} | {r['truncated']} | {r['medianMs']} ms |")

drive_dir = Path("/content/drive/MyDrive/dislop-colab")
if Path("/content/drive/MyDrive").exists():
    drive_dir.mkdir(parents=True, exist_ok=True)
    drive_out = drive_dir / "t3-local-slm-colab.json"
    drive_out.write_text(Path("/content/t3-local-slm-colab.json").read_text(encoding="utf-8"), encoding="utf-8")
    print("Synced to Google Drive:", drive_out)